In [2]:
import pandas as pd
import os

In [ ]:
#awk -F'\t' 'NR==1 || ($16==9606 && $17==9606)' BIOGRID-ALL-3.5.168.tab2.txt > BIOGRID-HUMAN.tab2.txt

In [6]:
# The file path to your data
file_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/BIOGRID-HUMAN.tab2.txt'

# Define column names based on your data structure
column_names = [
    'BioGRID Interaction ID', 'Entrez Gene Interactor A', 'Entrez Gene Interactor B',
    'BioGRID ID Interactor A', 'BioGRID ID Interactor B', 'Systematic Name Interactor A',
    'Systematic Name Interactor B', 'Official Symbol Interactor A', 'Official Symbol Interactor B',
    'Synonyms Interactor A', 'Synonyms Interactor B', 'Experimental System',
    'Experimental System Type', 'Author', 'Pubmed ID', 'Organism ID Interactor A',
    'Organism ID Interactor B', 'Throughput', 'Score', 'Modification', 'Phenotypes','Qualifications', 'Tags',
    'Source Database'
]

# Read the CSV file, using 'names' to specify column names
connection_all = pd.read_csv(file_path, sep='\t', names=column_names, header=0)
connection_df = connection_all[['Entrez Gene Interactor A', 'Entrez Gene Interactor B']]

/tmp/ipykernel_3581969/97206783.py:16: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  connection_all = pd.read_csv(file_path, sep='\t', names=column_names, header=0)


In [10]:
print(len(connection_df))
connection_df = connection_df.drop_duplicates()
print(len(connection_df))

432447
337750


In [11]:
connection_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/biograd_entrz_2019.txt', sep='\t', index=False, header=False)

In [9]:
len(connection_df),len(connection_df.drop_duplicates())

(432447, 337750)

In [13]:
### PPI features from node2vec
from pecanpy import pecanpy as node2vec

def Runnode2vec(filepath):
    n2v = node2vec.SparseOTF(p=1, q=1, workers=4, verbose=True)

    edge_list = n2v.read_edg(filepath, weighted=False, directed=False)
    emd = n2v.embed(dim=128, num_walks=10, walk_length=80, window_size=10, epochs=10)

    n2v_emd = pd.DataFrame(emd, n2v.nodes)

    n2v_emd.columns = ['network_' + str(col) for col in n2v_emd.columns]

    n2v_emd = n2v_emd.reset_index().rename(columns={"index":"entrz"})

    return n2v_emd

In [14]:
ppi_features = Runnode2vec('/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/biograd_entrz_2019.txt')

new_columns = ['entrz'] + [f'feature_{i}' for i, col in enumerate(ppi_features.columns) if col != 'entrz']
# Reorder the DataFrame so that 'string_id' is the first column
df_combined = ppi_features[['entrz'] + [col for col in ppi_features.columns if col != 'entrz']]
df_combined.columns = new_columns
df_combined.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/biograd_entrz_emb_2019.csv', index=False)

  0%|          | 0/174090 [00:00<?, ?it/s]

In [16]:
df_combined.shape, df_combined.columns

((17409, 129),
 Index(['entrz', 'feature_1', 'feature_2', 'feature_3', 'feature_4',
        'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9',
        ...
        'feature_119', 'feature_120', 'feature_121', 'feature_122',
        'feature_123', 'feature_124', 'feature_125', 'feature_126',
        'feature_127', 'feature_128'],
       dtype='object', length=129))

In [17]:
import mygene

def get_map_df(ensembl_ids,input_type):
    mg = mygene.MyGeneInfo()
    # Query mygene for UniProt and Entrez gene ID mappings
    results = mg.querymany(
        ensembl_ids,
        scopes=input_type,
        fields='uniprot,entrezgene',
        species='human'
    )

    results_df = pd.DataFrame(results)
    results_df = results_df[~results_df['entrezgene'].isna()]
    results_df['uniprot_ids'] = results_df['uniprot'].apply(
        lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
    results_df = results_df[~results_df['uniprot_ids'].isna()]
    results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]
    return results_df

bio_ids_map = get_map_df(df_combined['entrz'],'entrezgene')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
54 input query terms found no hit:	['9220', '26148', '11217', '147791', '79686', '284014', '23285', '6025', '440414', '117153', '373073


In [18]:
bio_ids_map

,query,_id,_score,entrezgene,uniprot,notfound,uniprot_ids
0,6416,6416,25.996578,6416,"{'Swiss-Prot': 'P45985', 'TrEMBL': ['R4GN68', ...",NaN,P45985
1,2318,2318,26.518696,2318,"{'Swiss-Prot': 'Q14315', 'TrEMBL': ['A0AAQ5BHM...",NaN,Q14315
2,84665,84665,26.518112,84665,"{'Swiss-Prot': 'Q86TC9', 'TrEMBL': ['A0A8I5KX0...",NaN,Q86TC9
3,88,88,25.607040,88,"{'Swiss-Prot': 'P35609', 'TrEMBL': ['A0A804HKG...",NaN,P35609
4,90,90,26.518112,90,"{'Swiss-Prot': 'Q04771', 'TrEMBL': ['C9JW28', ...",NaN,Q04771
...,...,...,...,...,...,...,...
17404,440957,440957,27.309893,440957,"{'Swiss-Prot': 'Q8WVI0', 'TrEMBL': ['F8W7Q2', ...",NaN,Q8WVI0
17405,144535,144535,25.996578,144535,"{'Swiss-Prot': 'Q96N23', 'TrEMBL': ['A0A0A0MTQ...",NaN,Q96N23
17406,23626,23626,26.518112,23626,"{'Swiss-Prot': 'Q9Y5K1', 'TrEMBL': ['Q5TCH6', ...",NaN,Q9Y5K1
17407,157777,157777,27.309893,157777,"{'Swiss-Prot': 'Q4G0Z9', 'TrEMBL': ['G3XAN3', ...",NaN,Q4G0Z9


In [19]:
ppi_set = set()
for values in bio_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)

string_ids = []
one2more = []
more2one = []  # to collect subdfs with multiple or zero matches

for uniport_ids in list(ppi_set):
    subdf = bio_ids_map[bio_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
    
    if len(subdf) == 1:
        if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
            one2more.append(subdf)
        else:
            string_ids.append(uniport_ids)
    else:
        more2one.append(subdf)

more2one_df = pd.concat(more2one, ignore_index=True)

In [ ]:
df_combined.rename(columns={'entrz':'string_id'},inplace=True)
df_combined = df_combined.set_index("string_id")

In [43]:
# Prepare list to store the results
aggregated_rows = []

# Iterate over each UniProt ID group
for protein_id, subdf in more2one_df.groupby('uniprot_ids'):
    # Get list of ENSP IDs
    ensp_ids = subdf['query'].tolist()

    # Select corresponding rows from ppi_emb where 'string_id' is in ensp_ids
    matched_ppi = df_combined[df_combined.index.isin(ensp_ids)]

    if not matched_ppi.empty:
        # Calculate the mean of all feature columns (exclude 'string_id')
        mean_features = matched_ppi.mean()

        # Create a new row with UniProt ID and the averaged features
        mean_features['string_id'] = protein_id

        # Add to the results list
        aggregated_rows.append(mean_features)

# Convert the list of Series into a DataFrame
aggregated_df = pd.DataFrame(aggregated_rows)

# Optional: Reorder columns to have 'uniprot_id' first
cols = ['string_id'] + [col for col in aggregated_df.columns if col != 'string_id']
aggregated_df = aggregated_df[cols]

# Step 1: Prepare ENSP IDs with '9606.' prefix
ensp_ids = [ensp_id for ensp_id in bio_ids_map[bio_ids_map['uniprot_ids'].isin(string_ids)]['query'].tolist()]

# Step 2: Select matching rows from ppi_emb
other_ppi = df_combined[df_combined.index.isin(ensp_ids)].copy()

# Step 3: Map 'string_id' back to 'uniprot_ids'
ensp_to_uniprot = bio_ids_map[bio_ids_map['uniprot_ids'].isin(string_ids)].set_index('query')['uniprot_ids'].to_dict()

# Apply mapping to refill 'string_id' with corresponding UniProt ID
other_ppi['string_id'] = other_ppi.index.map(lambda x: ensp_to_uniprot[x])

bio_emb_df = pd.concat([other_ppi, aggregated_df], ignore_index=True)

new_columns = ['string_id'] + [f'feature_{i}' for i, col in enumerate(bio_emb_df.columns) if col != 'string_id']

# Reorder the DataFrame so that 'string_id' is the first column
bio_emb_df = bio_emb_df[['string_id'] + [col for col in bio_emb_df.columns if col != 'string_id']]
bio_emb_df.columns = new_columns


In [45]:
bio_emb_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/biograd/uniport_biogrid_emb_2019.csv',index = False)